# Global Pollution Analysis and Energy Recovery (FINAL CORRECT VERSION)

Includes preprocessing, clustering (K-Means + Hierarchical), and neural network regression.

## Phase 1 – Data Preprocessing & Feature Engineering

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

df = pd.read_csv("Global_Pollution_Analysis.csv")
df.head()

In [ ]:
# Handle missing values
df = df.dropna()

In [ ]:
# Encode categorical features
le = LabelEncoder()
df['Country'] = le.fit_transform(df['Country'])
df['Year'] = le.fit_transform(df['Year'])

In [ ]:
# Feature Engineering
df['Energy_per_Capita'] = df['Energy_Consumption'] / df['Population']

# Pollution trend feature
df['Pollution_Index'] = df[['Air_Pollution','Water_Pollution','Soil_Pollution']].mean(axis=1)

In [ ]:
# Normalize data
scaler = StandardScaler()
cols = ['Air_Pollution','Water_Pollution','Soil_Pollution','CO2_Emissions','Energy_per_Capita','Pollution_Index']
df[cols] = scaler.fit_transform(df[cols])

## Phase 2 – Clustering

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

X_cluster = df[['Air_Pollution','Water_Pollution','Soil_Pollution','Energy_per_Capita']]

# Elbow Method
wcss = []
for i in range(1,10):
    kmeans = KMeans(n_clusters=i)
    kmeans.fit(X_cluster)
    wcss.append(kmeans.inertia_)

plt.plot(range(1,10), wcss)
plt.title("Elbow Method")
plt.xlabel("Clusters")
plt.ylabel("WCSS")
plt.show()

In [ ]:
# Apply KMeans
kmeans = KMeans(n_clusters=3)
df['KMeans_Cluster'] = kmeans.fit_predict(X_cluster)

plt.scatter(df['Pollution_Index'], df['Energy_Recovered'], c=df['KMeans_Cluster'])
plt.title("K-Means Clusters")
plt.show()

In [ ]:
# Hierarchical Clustering
import scipy.cluster.hierarchy as sch

plt.figure(figsize=(10,5))
sch.dendrogram(sch.linkage(X_cluster, method='ward'))
plt.title("Dendrogram")
plt.show()

In [ ]:
from sklearn.cluster import AgglomerativeClustering

hc = AgglomerativeClustering(n_clusters=3)
df['Hierarchical_Cluster'] = hc.fit_predict(X_cluster)

## Phase 3 – Neural Network (REGRESSION)

In [ ]:
X = df[['Air_Pollution','Water_Pollution','Soil_Pollution','CO2_Emissions','Energy_per_Capita','Pollution_Index']]
y = df['Energy_Recovered']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

model = Sequential()
model.add(Dense(16, activation='relu', input_dim=X.shape[1]))
model.add(Dense(8, activation='relu'))
model.add(Dense(1))

model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.fit(X_train, y_train, epochs=20)

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

y_pred = model.predict(X_test)

print("MSE:", mean_squared_error(y_test, y_pred))
print("MAE:", mean_absolute_error(y_test, y_pred))
print("R2:", r2_score(y_test, y_pred))

## Model Comparison

In [ ]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

print("Linear Regression R2:", r2_score(y_test, y_pred_lr))

## Insights


- Higher pollution levels tend to increase energy recovery potential.
- Clustering reveals groups of countries with similar pollution patterns.
- Neural networks outperform linear regression in capturing nonlinear relationships.
- Pollution trends over years impact recovery efficiency.
